# Optimización Logística en E-Commerce
### Módulo 2: Análisis Visual y Diagnóstico de Cuellos de Botella

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path

# Configuracion global corporativa para las graficas
pio.templates.default = "plotly_white"

# Definicion de rutas estandarizadas
BASE_DIR = Path.cwd().parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PATH_DATA = PROCESSED_DIR / "ecommerce_analytical_base.parquet"

# Carga de la sábana de datos analitica
df = pd.read_parquet(PATH_DATA)

## 1. Volumetría y Tendencia Histórica
Evaluación de la demanda a lo largo del tiempo para identificar el comportamiento general del mercado.

In [2]:
# Extraccion del periodo (Año-Mes)
df['order_month_period'] = df['order_purchase_timestamp'].dt.to_period('M').astype(str)

# Agrupacion temporal
df_trend = (
    df.groupby('order_month_period')
    .size()
    .reset_index(name='order_count')
)

fig_trend = px.line(
    df_trend,
    x='order_month_period',
    y='order_count',
    title='Tendencia Histórica de Pedidos Procesados',
    labels={'order_month_period': 'Mes-Año', 'order_count': 'Volumen de Pedidos'},
    markers=True
)

fig_trend.update_layout(xaxis_tickangle=-45)
fig_trend.show()

## 2. Estacionalidad Mensual
Identificación de picos de demanda estacionales agrupando el volumen transaccional por mes natural, independientemente del año.

In [3]:
# Extraccion del nombre del mes utilizando metodos nativos vectorizados
df['purchase_month'] = df['order_purchase_timestamp'].dt.month_name()

# Definicion de orden cronologico para evitar el ordenamiento alfabetico por defecto
months_order = ['January', 'February', 'March', 'April', 'May', 'June', 
                'July', 'August', 'September', 'October', 'November', 'December']
df['purchase_month'] = pd.Categorical(df['purchase_month'], categories=months_order, ordered=True)

df_seasonality = (
    df.groupby('purchase_month')
    .size()
    .reset_index(name='order_count')
)

fig_seasonality = px.bar(
    df_seasonality,
    x='purchase_month',
    y='order_count',
    title='Estacionalidad Agregada: Volumen de Pedidos por Mes',
    labels={'purchase_month': 'Mes del Año', 'order_count': 'Volumen Acumulado'},
    color_discrete_sequence=['#2C3E50']
)

fig_seasonality.show()

## 3. Diagnóstico de Ineficiencia Logística por Región
Análisis de los estados con mayores tiempos de demora promedio respecto a su fecha de entrega estimada.

In [4]:
# Filtrado, agrupacion y ordenamiento en una sola cadena de metodos
df_state_delay = (
    df[df['delivery_delay_days'] > 0]
    .groupby('customer_state', as_index=False)['delivery_delay_days']
    .mean()
    .sort_values(by='delivery_delay_days', ascending=True) # Ascendente para que Plotly coloque el mayor arriba
    .tail(10) # Top 10 peores estados
)

fig_state_delay = px.bar(
    df_state_delay,
    x='delivery_delay_days',
    y='customer_state',
    orientation='h',
    title='Top 10 Estados con Mayor Retraso Promedio de Entrega',
    labels={'customer_state': 'Estado (Código)', 'delivery_delay_days': 'Días de Retraso Promedio'},
    color='delivery_delay_days',
    color_continuous_scale='Reds'
)

fig_state_delay.show()

## 4. Diagrama de Pareto: Análisis de Causas Raíz
Aplicación del principio 80/20 para identificar qué categorías de productos concentran el mayor volumen de insatisfacción del cliente debido a retrasos logísticos.

In [5]:
# Extraccion de pedidos que sufrieron retrasos Y generaron quejas
df_pareto = (
    df[(df['delivery_delay_days'] > 0) & (df['review_score_category'] == 'negative')]
    .groupby('product_category_name_english', as_index=False)
    .size()
    .rename(columns={'size': 'complaint_count'})
    .sort_values(by='complaint_count', ascending=False)
)

# Calculo de la suma acumulada porcentual
df_pareto['cumulative_percent'] = (df_pareto['complaint_count'].cumsum() / df_pareto['complaint_count'].sum()) * 100

# Limitamos a las 15 categorias principales para claridad visual
df_pareto_top = df_pareto.head(15)

# Construccion del grafico de doble eje
fig_pareto = go.Figure()

# Eje Y Primario (Barras de conteo)
fig_pareto.add_trace(go.Bar(
    x=df_pareto_top['product_category_name_english'],
    y=df_pareto_top['complaint_count'],
    name='Volumen de Quejas',
    marker_color='#34495E'
))

# Eje Y Secundario (Linea acumulada)
fig_pareto.add_trace(go.Scatter(
    x=df_pareto_top['product_category_name_english'],
    y=df_pareto_top['cumulative_percent'],
    name='% Acumulado',
    mode='lines+markers',
    line=dict(color='#E74C3C', width=3),
    yaxis='y2'
))

# Configuracion de la arquitectura del grafico
fig_pareto.update_layout(
    title='Diagrama de Pareto: Concentración de Quejas Logísticas por Categoría',
    xaxis_title='Categoría de Producto',
    yaxis_title='Número de Quejas',
    yaxis2=dict(
        title='Porcentaje Acumulado (%)',
        overlaying='y',
        side='right',
        range=[0, 105] # Margen superior para visualizar el 100%
    ),
    legend=dict(x=0.01, y=1.1, orientation='h'),
    margin=dict(r=50)
)

fig_pareto.show()